# 04 · vLLM 的调度循环：Continuous Batching

第 02 章证明了单个请求喂不饱 GPU。这一章解决下一个问题：**怎么把多个请求拼在一起跑。**

但这一章有个额外的目标：**让你学完之后打开 vLLM 源码能直接对上号。** 所以下面的类名、职责划分、主循环结构都刻意照搬了 vLLM V1。

## 三层架构（先记住这张图）

```
EngineCore.step()                        ←→ vllm/v1/engine/core.py
  ├─ Scheduler.schedule()                ←→ vllm/v1/core/sched/scheduler.py
  │    决定这一轮跑哪些请求、各自几个 token      （不碰显存，不碰模型）
  ├─ ModelRunner.execute_model()         ←→ vllm/v1/worker/gpu_model_runner.py
  │    把排好的请求拼成 batch，跑模型，采样        （不碰调度策略）
  └─ Scheduler.update_from_output()
       把结果写回请求状态，处理完成与回收
```

**这个职责划分是本章最值钱的东西。** 面试被问"vLLM 架构"，把这三层和各自的边界讲清楚，比背模块名有用得多。后面所有优化——chunked prefill、前缀缓存、抢占、投机解码——都是在这三步里插桩。

In [ ]:
# ===== 引导单元：环境检查 + 测量工具 + MiniGPT（每章自带，直接运行）=====
# 说明：本单元在每个 notebook 里都有一份完整副本，目的是让任何一个 notebook
#       都能在 Colab 里零配置独立运行。想改模型结构，请改 tools/build_notebooks.py
#       里的 SETUP_CODE，然后重跑编译脚本。
#
# 架构对齐：下面这套推理核心刻意模仿了 vLLM V1 的模块划分与命名，
#   详见 docs/vllm-mapping.md 的对照表。
#       EngineCore.step()           ←→ vllm/v1/engine/core.py
#         ├─ Scheduler.schedule()   ←→ vllm/v1/core/sched/scheduler.py
#         ├─ ModelRunner.execute_model() ←→ vllm/v1/worker/gpu_model_runner.py
#         └─ Scheduler.update_from_output()
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MiniGPT 只有 2700 万参数，用 float16 跑在 GPU 上；CPU 上 float16 很慢，用 float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


def sync():
    """GPU 是异步执行的，计时前必须同步，否则测到的是下发时间不是执行时间。"""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def bench(fn, warmup=3, iters=10):
    """返回单次调用的平均耗时（毫秒）。warmup 用来排除首次 kernel 编译等开销。"""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters * 1000.0


def peak_mem_mb():
    """当前 CUDA 峰值显存占用（MB）。"""
    if DEVICE != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 2


def reset_peak():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()


class Config:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=4, n_head=6, n_embd=384):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head


class CausalSelfAttention(nn.Module):
    """因果自注意力，支持 KV cache。

    past_kv 传入历史的 (k, v)，本步只为新 token 计算 Q/K/V，然后拼在历史后面。
    返回 (输出, 更新后的 (k, v))，其中 k/v 的 shape 是 (B, n_head, 总长度, head_dim)。
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x, past_kv=None, attn_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

        S = k.size(2)  # 总长度 = 历史 + 本步新增
        if attn_mask is None:
            # 默认因果掩码：本步第 i 个 query 的绝对位置是 S-T+i，只能看见 <= 它的 key
            mask = torch.ones(T, S, device=x.device).tril(diagonal=S - T).bool()
        else:
            # 外部传入的掩码，用于一个 batch 里混合不同进度的序列（第 04、06 章）
            mask = attn_mask
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, past_kv=None, attn_mask=None):
        h, present = self.attn(self.ln_1(x), past_kv, attn_mask)
        x = x + h
        x = x + self.mlp(self.ln_2(x))
        return x, present


class MiniGPT(nn.Module):
    """极简 GPT，结构与 Llama 同源：pre-norm + 因果注意力 + 4 倍扩张 MLP + 权重共享。

    与 Llama 的两处差异：
      - 用可学习位置编码代替 RoPE（简化实现，不影响调度实验的结论）
      - 没有 GQA（本仓库是 MHA，第 03 章会手工比较两者的 KV cache 大小）
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # 权重共享，省一份 embedding 参数

        def init(m):
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        self.apply(init)

    def forward(self, idx, past_kvs=None, pos_offset=0, attn_mask=None):
        """idx: (B, T) 的 token id。

        past_kvs: 长度等于层数的列表，每项是 (k, v)；None 表示从零开始（prefill）。
        pos_offset: 本次输入的第一个 token 的绝对位置。传 int 表示整个 batch 用同一个
                    偏移；传 shape (B,) 的张量表示每条序列各用各的偏移——当 batch 里
                    混合了不同进度的请求时必须这样传。
        attn_mask: 可选的自定义注意力掩码，用于屏蔽填充位。
        """
        B, T = idx.shape
        if torch.is_tensor(pos_offset):
            pos = pos_offset.view(B, 1) + torch.arange(T, device=idx.device)[None, :]
        else:
            pos = torch.arange(pos_offset, pos_offset + T, device=idx.device)[None, :].expand(B, T)
        x = self.wte(idx) + self.wpe(pos)

        presents = []
        for i, blk in enumerate(self.blocks):
            past = None if past_kvs is None else past_kvs[i]
            x, present = blk(x, past, attn_mask)
            presents.append(present)
        return self.lm_head(self.ln_f(x)), presents

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def build_model(seed=0, device=DEVICE, dtype=DTYPE, **kw):
    torch.manual_seed(seed)
    cfg = Config(**kw)
    model = MiniGPT(cfg).to(device=device, dtype=dtype)
    return model.eval()


@torch.no_grad()
def generate_naive(model, idx, max_new_tokens):
    """不用 KV cache：每一步都把完整序列重新算一遍（O(n^2) 重算）。"""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """用 KV cache：prompt 只 prefill 一次，之后每步只喂 1 个 token。"""
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [nxt]
    pos = idx.size(1)
    for _ in range(max_new_tokens - 1):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat([idx] + out, dim=1)


def kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=2):
    """KV cache 字节数。注意是 2（K 和 V 各一份）。"""
    return 2 * n_layer * n_kv_head * head_dim * seq_len * batch * dtype_bytes


# ========== 以下是模仿 vLLM V1 架构的推理核心 ==========


class Request:
    """对应 vllm/v1/request.py 的 Request。

    num_computed_tokens 是 vLLM 里最核心的一个字段：它记录这条请求已经有
    多少 token 的 KV 被算过。prefill、chunked prefill、前缀缓存命中——
    三种看起来完全不同的场景，在 vLLM 里都只是「把 num_computed_tokens 往前推」。
    理解这一点，chunked prefill 就不再是独立机制，而是这个字段的自然结果。
    """

    def __init__(self, request_id, prompt_token_ids, max_tokens):
        self.request_id = request_id
        self.prompt_token_ids = list(prompt_token_ids)
        self.max_tokens = max_tokens
        self.output_token_ids = []
        self.num_computed_tokens = 0
        self.status = "waiting"      # waiting / running / finished
        # 本仓库简化：直接把 KV 张量挂在请求上。
        # 真实 vLLM 不这么做——请求只持有 block_table，物理 block 由 KVCacheManager 管（第 05 章）。
        self.past = None

    @property
    def num_prompt_tokens(self):
        return len(self.prompt_token_ids)

    def all_token_ids(self):
        return self.prompt_token_ids + self.output_token_ids

    def num_tokens_to_schedule(self):
        """还欠多少 token 没算：prefill 阶段是剩余 prompt 长度，decode 阶段是 1。"""
        if self.num_computed_tokens < self.num_prompt_tokens:
            return self.num_prompt_tokens - self.num_computed_tokens
        return 1

    @property
    def is_finished(self):
        return len(self.output_token_ids) >= self.max_tokens

    def __repr__(self):
        return (f"Request({self.request_id}, computed={self.num_computed_tokens}"
                f"/{self.num_prompt_tokens}, out={len(self.output_token_ids)}"
                f"/{self.max_tokens}, {self.status})")


class SchedulerOutput:
    """对应 vllm/v1/core/sched/output.py 的 SchedulerOutput。

    调度与执行之间唯一的接口。真实 vLLM 里这个结构还包含 block 分配结果、
    抢占列表等字段，这里只保留最必要的两个。
    """

    def __init__(self, scheduled_reqs, num_scheduled_tokens):
        self.scheduled_reqs = scheduled_reqs
        self.num_scheduled_tokens = num_scheduled_tokens   # {request_id: n}

    def __len__(self):
        return len(self.scheduled_reqs)


class Scheduler:
    """对应 vllm/v1/core/sched/scheduler.py 的 Scheduler。

    职责边界是这个架构里最值得学的一点：Scheduler 只决定
    「这一轮跑哪些请求、各自跑几个 token」，它既不碰显存也不碰模型。

        显存分配 → KVCacheManager（第 05 章）
        真正计算 → ModelRunner

    三个模块分离，才能各自独立替换实现。面试被问「说说 vLLM 的架构」时，
    先把这个职责划分讲清楚，比背模块名有用得多。
    """

    def __init__(self, max_num_seqs=8, max_num_batched_tokens=2048):
        self.waiting = []
        self.running = []
        self.finished = []
        self.max_num_seqs = max_num_seqs
        # 这个预算就是 chunked prefill 的开关：调小它，长 prompt 自然被切成多轮（第 06 章）
        self.max_num_batched_tokens = max_num_batched_tokens
        self.step_id = 0

    def add_request(self, req):
        self.waiting.append(req)

    def has_unfinished(self):
        return bool(self.waiting or self.running)

    def schedule(self):
        scheduled, num_tokens = [], {}
        budget = self.max_num_batched_tokens

        # 第一优先：正在跑的请求。已进 decode 的排 1 个 token；
        # 还在做 chunked prefill 的按剩余量排，但受 budget 限制。
        for req in list(self.running):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n

        # 第二优先：从队列里补新请求进来做 prefill
        for req in list(self.waiting):
            if budget <= 0 or len(scheduled) >= self.max_num_seqs:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n
            self.waiting.remove(req)
            req.status = "running"
            self.running.append(req)

        self.step_id += 1
        return SchedulerOutput(scheduled, num_tokens)

    def update_from_output(self, sched_out, sampled):
        """对应 vLLM 的 update_from_output：写回采样结果，处理完成与回收。

        本轮被调度但没产生 token 的请求（比如 chunked prefill 的中间块）
        不会出现在 sampled 里，它们保持 running，下一轮继续。
        """
        for req in sched_out.scheduled_reqs:
            if req.request_id not in sampled:
                continue
            req.output_token_ids.append(sampled[req.request_id])
            if req.is_finished:
                req.status = "finished"
                if req in self.running:
                    self.running.remove(req)
                self.finished.append(req)
                req.past = None      # 简化回收；真实 vLLM 走 KVCacheManager.free()


class ModelRunner:
    """对应 vllm/v1/worker/gpu_model_runner.py 的 GPUModelRunner。

    职责：把 Scheduler 排好的一批请求拼成一次前向，返回新采样的 token。

    与真实 vLLM 的差距（要如实知道）：
      · vLLM 用 block_table 让每条序列的 KV 物理上不连续，所以不需要填充；
        这里用「右填充 + 逐序列掩码」对齐，会浪费显存——第 05 章解决。
      · vLLM 会把 prefill 和 decode 混在同一个 batch 里跑；这里分成两组处理，
        纯粹是为了让代码可读，结论不受影响。
      · 输入准备、CUDA graph、attention metadata 这些都被省掉了。
    """

    def __init__(self, model):
        self.model = model

    @torch.no_grad()
    def _run_decode_batch(self, reqs):
        """把一批进度不同的 decode 请求拼成一次前向。"""
        B = len(reqs)
        lens = [r.num_computed_tokens for r in reqs]
        Lmax = max(lens)
        n_layer = self.model.cfg.n_layer

        padded = []
        for layer in range(n_layer):
            ks, vs = [], []
            for r in reqs:
                k, v = r.past[layer]
                pad = Lmax - k.size(2)
                if pad:
                    k = F.pad(k, (0, 0, 0, pad))
                    v = F.pad(v, (0, 0, 0, pad))
                ks.append(k)
                vs.append(v)
            padded.append((torch.cat(ks, 0), torch.cat(vs, 0)))

        # 逐序列掩码：真实历史 [0, L_i) + 新 token 落在下标 Lmax
        S = Lmax + 1
        mask = torch.zeros(B, 1, 1, S, dtype=torch.bool, device=DEVICE)
        for i, r in enumerate(reqs):
            mask[i, 0, 0, : lens[i]] = True
            mask[i, 0, 0, Lmax] = True

        ids = torch.tensor([[r.all_token_ids()[r.num_computed_tokens]] for r in reqs],
                           device=DEVICE)
        pos = torch.tensor(lens, device=DEVICE)
        logits, past = self.model(ids, past_kvs=padded, pos_offset=pos, attn_mask=mask)

        sampled = {}
        for i, r in enumerate(reqs):
            rebuilt = []
            for layer in range(n_layer):
                k_all, v_all = past[layer]
                k = torch.cat([k_all[i:i + 1, :, : lens[i]],
                               k_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                v = torch.cat([v_all[i:i + 1, :, : lens[i]],
                               v_all[i:i + 1, :, Lmax:Lmax + 1]], dim=2)
                rebuilt.append((k, v))
            r.past = rebuilt
            r.num_computed_tokens += 1
            sampled[r.request_id] = int(logits[i, -1].argmax(-1).item())
        return sampled

    @torch.no_grad()
    def execute_model(self, sched_out):
        decode_reqs, prefill_reqs = [], []
        for r in sched_out.scheduled_reqs:
            # 判断依据是「prompt 算完了没有」，而不是「本轮排了几个 token」
            if r.num_computed_tokens >= r.num_prompt_tokens:
                decode_reqs.append(r)
            else:
                prefill_reqs.append(r)

        sampled = {}
        if decode_reqs:
            sampled.update(self._run_decode_batch(decode_reqs))

        for r in prefill_reqs:
            n = sched_out.num_scheduled_tokens[r.request_id]
            start = r.num_computed_tokens
            chunk = r.all_token_ids()[start:start + n]
            toks = torch.tensor([chunk], device=DEVICE)
            logits, past = self.model(toks, past_kvs=r.past, pos_offset=start)
            r.past = past
            r.num_computed_tokens += len(chunk)
            # 只有 prompt 全部算完，才能采样第一个输出 token
            if r.num_computed_tokens >= r.num_prompt_tokens:
                sampled[r.request_id] = int(logits[:, -1].argmax(-1).item())
        return sampled


class EngineCore:
    """对应 vllm/v1/engine/core.py 的 EngineCore。

    整个 vLLM 的推理服务就跑在这三步上：

        schedule()            决定这一轮跑什么
        execute_model()       跑模型
        update_from_output()  把结果写回请求状态

    读懂这个循环你就抓住了 vLLM 的主干。后面所有优化——chunked prefill、
    前缀缓存、抢占、投机解码——都是在这三步里插桩。
    """

    def __init__(self, model, scheduler=None):
        self.scheduler = scheduler or Scheduler()
        self.runner = ModelRunner(model)
        self.step_id = 0
        self.steps = 0

    def step(self):
        sched_out = self.scheduler.schedule()
        if len(sched_out) == 0:
            return None
        sampled = self.runner.execute_model(sched_out)
        self.scheduler.update_from_output(sched_out, sampled)
        self.step_id += 1
        self.steps += 1
        return sampled

    def run(self, max_steps=10000):
        while self.scheduler.has_unfinished() and self.steps < max_steps:
            self.step()
        return self.steps


print(f"引导单元加载完成 | device={DEVICE} dtype={DTYPE} torch={torch.__version__}")
# ===== 引导单元结束 =====

## 一、先跑通一次，看清这个循环

引导单元里已经提供了 `Request`、`Scheduler`、`ModelRunner`、`EngineCore`。先做一次最小实验，把每一步的调度决策打出来。

In [ ]:
model = build_model(block_size=4096)


def make_workload(n=32, prompt_len=64, lo=4, hi=65, vocab=50257, seed=0):
    """注意 prompt 是 Python list 不是张量——Request 持有的是 token id 序列，
    真正的张量由 ModelRunner 在跑模型时临时拼。这也是 vLLM 的做法。"""
    g = torch.Generator().manual_seed(seed)
    reqs = []
    for i in range(n):
        prompt = torch.randint(0, vocab, (prompt_len,), generator=g).tolist()
        max_tokens = int(torch.randint(lo, hi, (1,), generator=g).item())
        reqs.append(Request(f"req{i}", prompt, max_tokens))
    return reqs


# 6 条请求，batch 上限 4，看调度器怎么在 waiting 和 running 之间搬人
engine = EngineCore(model, Scheduler(max_num_seqs=4, max_num_batched_tokens=4096))
for r in make_workload(n=6, prompt_len=16, lo=2, hi=6, seed=1):
    engine.scheduler.add_request(r)

print(f"{'step':>5}{'running':>9}{'waiting':>9}   明细 (req: 已算token/prompt + 已生成)")
print("-" * 84)
while engine.scheduler.has_unfinished():
    engine.step()
    s = engine.scheduler
    detail = "  ".join(f"{r.request_id}:{r.num_computed_tokens}/{r.num_prompt_tokens}"
                       f"+{len(r.output_token_ids)}" for r in s.running)
    print(f"{s.step_id:>5}{len(s.running):>9}{len(s.waiting):>9}   {detail}")

print()
print("每个请求的最终状态：")
for r in engine.scheduler.finished:
    print(" ", r)

观察这几点，它们就是 continuous batching 的全部内容：

1. **每轮结束就重新组批**。某个请求完成后立刻从 `running` 移除，`waiting` 里的新请求马上补进来——不需要等整批跑完。
2. **`num_computed_tokens` 一路往前推**。prefill 时它一次性涨到 prompt 长度；之后每轮 +1（decode）。这个字段是理解 vLLM 的钥匙。
3. **新请求在前几轮占用更多 token 预算**，因为要先把 prompt 算完。

## 二、读懂 `EngineCore.step()`

主循环只有三步，但每一步的边界都很清晰：

```python
def step(self):
    sched_out = self.scheduler.schedule()                   # 1. 决定跑什么
    sampled   = self.runner.execute_model(sched_out)        # 2. 跑模型
    self.scheduler.update_from_output(sched_out, sampled)   # 3. 写回状态
```

**为什么要把调度和执行分开？** 因为它们的变更频率完全不同：

| 模块 | 多久改一次 | 改什么 |
|---|---|---|
| Scheduler | 很频繁 | 调度策略、优先级、抢占规则 |
| ModelRunner | 很少改 | attention 后端、CUDA graph、量化 kernel |

揉在一起的话，调一个调度策略就要动模型执行代码，风险极高。分开之后，第 06 章要演示的 chunked prefill 只需要改 Scheduler 的一个预算参数。

### 一个刻意保留的差异

`ModelRunner` 把请求分成了 `decode_reqs` 和 `prefill_reqs` 两组分别处理。真实 vLLM 是把它们**混在同一个 batch 里**跑的，这才是 chunked prefill 能成立的前提。

这里分开处理纯粹是为了让代码能读懂——本章的实验结论（调度层面的差异）不受影响。真实实现里混批需要 block table，那是第 05 章的内容。

## 三、对照组：static batching

现在实现一个 static 版本的调度器。**注意：只改 `schedule()` 一个方法，执行和更新完全复用。** 这正是上面那个职责划分的价值——换调度策略不需要碰其他模块。

In [ ]:
class TracedScheduler(Scheduler):
    """加一层记录，用来统计每步的批次占用情况。"""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.trace = []

    def _record(self, out):
        self.trace.append({
            "step": self.step_id,
            "n_scheduled": len(out.scheduled_reqs),
            "n_running": len(self.running),
            "n_waiting": len(self.waiting),
            "tokens": sum(out.num_scheduled_tokens.values()),
        })
        return out

    def schedule(self):
        return self._record(super().schedule())


class StaticScheduler(TracedScheduler):
    """对照组：静态分组。一组全部跑完，才允许下一组进来。

    这就是传统 batching 的做法——像坐满一车人才发车的班车，
    而不是到站就上人下人的公交车。
    """

    def __init__(self, group_size=8, **kwargs):
        super().__init__(**kwargs)
        self.group_size = group_size

    def schedule(self):
        # 组内还有人就直接跑；整组清空了才开新的一组
        if not self.running and self.waiting:
            while self.waiting and len(self.running) < self.group_size:
                req = self.waiting.pop(0)
                req.status = "running"
                self.running.append(req)

        scheduled, num_tokens = [], {}
        budget = self.max_num_batched_tokens
        for req in list(self.running):
            if budget <= 0:
                break
            n = min(req.num_tokens_to_schedule(), budget)
            scheduled.append(req)
            num_tokens[req.request_id] = n
            budget -= n

        self.step_id += 1
        return self._record(SchedulerOutput(scheduled, num_tokens))


print("StaticScheduler 只重写了 schedule()，其余全部继承。")
print("这就是把调度和执行分开之后能拿到的东西。")

## 四、对比实验

In [ ]:
def run_engine(sched_cls, reqs, **kwargs):
    engine = EngineCore(model, sched_cls(**kwargs))
    for r in reqs:
        engine.scheduler.add_request(r)
    t0 = time.perf_counter()
    engine.run()
    dt = time.perf_counter() - t0
    return engine, dt


N, BATCH = 32, 8

static_engine, static_dt = run_engine(StaticScheduler, make_workload(N, seed=1),
                                      group_size=BATCH, max_num_batched_tokens=4096)
cont_engine, cont_dt = run_engine(TracedScheduler, make_workload(N, seed=1),
                                  max_num_seqs=BATCH, max_num_batched_tokens=4096)


def occupancy(engine, capacity):
    tr = engine.scheduler.trace
    return sum(t["n_running"] for t in tr) / len(tr) / capacity


total_tokens = sum(r.max_tokens for r in cont_engine.scheduler.finished)

print(f"{N} 条请求，batch 上限 {BATCH}，输出长度 4~64 随机\n")
print(f"{'调度方式':<22}{'总步数':>9}{'批次占用率':>12}{'耗时(s)':>10}{'吞吐(t/s)':>12}")
print("-" * 66)
print(f"{'static batching':<22}{static_engine.steps:>9}{occupancy(static_engine, BATCH):>11.1%}"
      f"{static_dt:>10.2f}{total_tokens / static_dt:>12,.0f}")
print(f"{'continuous batching':<22}{cont_engine.steps:>9}{occupancy(cont_engine, BATCH):>11.1%}"
      f"{cont_dt:>10.2f}{total_tokens / cont_dt:>12,.0f}")

### 结果解读

两个指标的改善来自同一个机制：

- **批次占用率**：static 会明显低于 continuous。因为一组里最短的输出 4 个 token、最长的 60 个，短的那条做完之后槽位就空着，直到整组跑完才换人。
- **总步数**：每一步都要跑一次完整前向，步数少意味着端到端更快。continuous 的槽位几乎不空转，同样的工作量需要更少步数。

**收益大小取决于输出长度的方差。** 如果所有请求输出长度都一样，static 的槽位永远不会空转，两者几乎没有差别。能说出这一点，说明你理解的是机制而不是结论。

### 代价

收益不是白来的，三个代价都要知道：

1. **延迟变得不可预测**：你不知道自己的请求会和谁拼在一起，p99 反而更难控制。线上必须用优先级或分池隔离。
2. **KV 管理复杂化**：batch 里每条序列进度不同、长度不同。本章的 `ModelRunner` 用"右填充 + 逐序列掩码"硬对齐，会浪费显存——这就是第 05 章 PagedAttention 要解决的问题。
3. **调度开销**：每轮都要重新组批，调度器本身也有成本。真实 vLLM 为此把调度逻辑写得很精细（优先级、抢占、token 预算分配）。

## 五、对照表：这个 lab ↔ vLLM 源码

学到这里，这些名字你应该都能对应上了。路径以 vLLM V1 为准，版本间会移动，用 `rg` 定位最可靠。

| 本章的东西 | vLLM 里的对应物 | 怎么找 | 差异 |
|---|---|---|---|
| `EngineCore.step()` | `EngineCore.step()` | `rg "def step" vllm/v1/engine/core.py` | 本 lab 同步执行，vLLM 异步 + 多进程 |
| `Scheduler.schedule()` | `Scheduler.schedule()` | `rg "def schedule" vllm/v1/core/sched/` | vLLM 有优先级、抢占、`skipped_waiting` |
| `SchedulerOutput` | `SchedulerOutput` | `vllm/v1/core/sched/output.py` | vLLM 还带 block 分配结果、preempted 列表 |
| `Request.num_computed_tokens` | 同名字段 | `vllm/v1/request.py` | 语义完全一致 |
| `Scheduler.running / waiting` | 同名字段 | `vllm/v1/core/sched/scheduler.py` | vLLM 还有 `skipped_waiting` |
| `ModelRunner.execute_model()` | `GPUModelRunner.execute_model()` | `vllm/v1/worker/gpu_model_runner.py` | vLLM 要做 InputBatch、CUDA graph、metadata |
| `max_num_batched_tokens` | **同名参数** | `vllm/config.py` → `SchedulerConfig` | 语义一模一样 |
| `max_num_seqs` | **同名参数** | `vllm/config.py` → `SchedulerConfig` | 一模一样 |

**两个参数的名字完全一致，这不是巧合**——它们就是你启动 vLLM 时能传的命令行参数。今天在 lab 里调它们观察到的现象，明天在真实服务上改它们会得到同样的结果。

## 六、面试话术

**问：continuous batching 相比 static batching 的收益来自哪？**

按这个顺序答：

1. **机制**：static 整批跑完才换人，短请求完成后槽位空转；continuous 每轮迭代重新组批，完成的立刻走、排队的立刻进。在 vLLM 里就是 `Scheduler` 的 `running` 和 `waiting` 两个队列在搬人。
2. **收益取决于什么**：输出长度分布的**方差**。方差越大，static 浪费越严重；长度整齐时两者几乎没差别。
3. **代价**：延迟不可预测、KV 管理复杂（催生了 PagedAttention）、调度本身有开销。
4. **架构视角（加分项）**：vLLM 把调度、显存、执行拆成 `Scheduler` / `KVCacheManager` / `ModelRunner` 三个模块，`EngineCore.step()` 是它们的胶水。这个划分让换调度策略不用碰模型执行代码——我在实验里只重写了一个 `schedule()` 方法就做出了 static batching 的对照组。

第 4 条是把"我懂概念"升级成"我读过源码"的关键。**能说出模块边界以及为什么这样划，比把机制复述一遍有说服力得多。**

**作业**

1. 把 `make_workload` 的输出长度范围改成 `lo=60, hi=65`（差异很小），重跑实验。两种方式的差距消失了吗？
2. 把 `max_num_seqs` 从 8 改成 32（等于一次放进所有请求），continuous 会退化成什么？
3. 给 `Scheduler` 加一个优先级：让 `max_tokens` 小的请求优先调度。观察完成时间的分布变化。（提示：vLLM 里有 `priority` 参数和 `SchedulingPolicy`）

**下一章**：解决本章留下的显存浪费——用分页的方式管理 KV cache。